In [ ]:
from PIL import Image
import pytesseract
import re

# Archivo de entrada (imagen JPG)
imagen = "musical/1968/Screenshot_2026-01-16-20-26-02-396_com.facebook.katana.jpg"

# Archivo de salida (texto limpio)
salida = "canciones_limpias.txt"

# 1. Extraer texto desde la imagen usando OCR
#texto_ocr = pytesseract.image_to_string(Image.open(imagen), lang="spa")
texto_ocr = pytesseract.image_to_string(Image.open(imagen))

# 2. Patrones flexibles para detectar canciones y autores
patrones = [
    r'["“](.+?)["”]\s*[-—=]\s*(.+)',   # "Título" — Autor
    r'(.+?)\s*[-—=]\s*(.+)',           # Título — Autor (sin comillas)
    r'(.+?)\s{2,}(.+)'                 # Título   Autor (OCR con espacios múltiples)
]

def limpiar_titulo(t):
    t = t.strip()
    t = t.strip('"“”')  # quitar comillas raras
    return t

def limpiar_autor(a):
    return a.strip()

resultados = []

# 3. Procesar línea por línea
for linea in texto_ocr.splitlines():
    linea = linea.strip()

    # Eliminar viñetas, números y símbolos iniciales
    linea = re.sub(r'^[\d\.\)\(]+', '', linea).strip()

    for patron in patrones:
        match = re.search(patron, linea)
        if match:
            titulo = limpiar_titulo(match.group(1))
            autor = limpiar_autor(match.group(2))

            # Filtrar basura OCR
            if len(titulo) > 1 and len(autor) > 1:
                resultados.append(f"{titulo} - {autor}")
            break

# 4. Ordenar y eliminar duplicados
resultados = sorted(set(resultados))

# 5. Guardar archivo limpio
with open(salida, "w", encoding="utf-8") as f:
    f.write("\n".join(resultados))

print("Archivo generado:", salida)
